# 04 - Evaluation

**Question this notebook answers:** does the RAG system retrieve the right evidence, stay grounded in it, and behave safely -- and is that better than asking the LLM alone?

```
01 Data + KB  ->  02 Build index  ->  03 RAG pipeline  ->  04 Evaluation
                                                            (you are here)
```

## What is measured

| Layer | Question | Metric |
|---|---|---|
| Generation | Does the answer contain the key facts? | **Key-fact coverage** (only questions that should be answered) |
| Behaviour | Does it answer, abstain, redirect or refuse when it should? | **Pass rate per behaviour**, **abstention recall and precision**, **over-refusal** |

"Declined" means the system did not answer from the sources: it abstained (the evidence was too weak), gave the emergency message, or redirected a personal-advice question. The abstention metrics are computed on declined answers, so a question that should not be answered and was redirected counts as correct, and an answerable question that was redirected counts as over-refusal.

## How it is set up

* **Questions:** `data/eval/questions.csv`, split into dev and test by the `split` column.
* **Judge:** a different model from the generator (`config.JUDGE_MODEL` vs `config.LLM_MODEL`), both on Groq with the same `GROQ_API_KEY`. Every verdict is saved in `reports/`.
* **LLM-only baseline:** the generator model asked the same questions with no sources (`BASELINE_SYSTEM` in `src/evaluation.py`). It is regenerated and re-judged on every run, so the baseline and the RAG system are always scored on the same questions, labels and key facts.
* **Threshold first, then the run.** Section 1 finds the best rerank score of every dev question (no LLM calls), chooses the abstention threshold from those scores and saves it. Only then is the pipeline run, so every table uses the calibrated threshold.
* **Dev / test discipline:** the threshold is chosen on dev. The test split is run once, with the threshold frozen.

## What the results can and cannot show

* Most questions are about common health topics that a large LLM already knows, so the baseline scores high on key-fact coverage. **Do not expect RAG to beat it on coverage.** RAG should add answers that stay inside the sources, visible citations, and an explicit "not enough information" instead of a confident guess.
* Medicines are **out of scope** (see notebook 01). Questions about named drugs are expected to be declined.
* Key facts were not reviewed by a clinician, and answers are graded by an LLM.
* The safety rules in `src/safety.py` were written while looking at some of these questions, so safety numbers on this set are optimistic. `tests/test_safety.py` holds separate cases.

In [ ]:
import json
import sys
from datetime import date
from pathlib import Path

import torch 

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config, safety
from src.evaluation import (
    abstention_metrics, choose_threshold, comparison_table, llm_only_answers,
    make_judge, md_table, score_behavior, score_key_facts, summarize,
)
from src.generator import make_client
from src.pipeline import load_pipeline

REPORTS = config.REPORTS_DIR
RUN_TEST = False  # set to True ONCE, after the threshold is frozen, to evaluate the test split

# Notebook 4 now regenerates the LLM-only baseline on every run (not just under RUN_TEST).
# If the generator key is exhausted for the day, the first 429 is usually a TPD limit on
# the generator model during the dev baseline cell, not during the RAG grading pass.
# When that happens, rerun only the baseline cell after the limit resets; do not rerun the
# whole notebook and burn the remaining quota on RAG grading first.

client = make_client()
judge = make_judge(make_client(max_retries=8))   # the judge makes hundreds of calls, so it waits out rate limits for longer

# These helper functions retry *transient* generator 429s a bounded number of times.
# Tokens-per-day exhaustion is raised immediately and clearly instead of being retried forever
# (that is the failure notebook 4 now hits during the regenerated LLM-only baseline cell).
# The judge client still relies on the OpenAI SDK's own max_retries; it is not wrapped here.

df_eval = pd.read_csv(config.EVAL_PATH)
df_dev = df_eval[df_eval["split"] == "dev"].reset_index(drop=True)
df_test = df_eval[df_eval["split"] == "test"].reset_index(drop=True)

print("generator:", config.LLM_MODEL, "| judge:", judge.model)
print(f"dev: {len(df_dev)} questions | test: {len(df_test)} questions")

## 1. Choosing the abstention threshold

The pipeline answers only if the best passage's rerank score is at least a threshold. Rather than guess a number, choose it on the dev questions. Retrieval and reranking need no LLM, so this step only loads the models and scores every question.

* **should answer:** `expected_behavior == "answer"`
* **should abstain:** `expected_behavior == "insufficient_evidence"` (unknown conditions, live information, medicines, ...)

Questions that the safety rules answer before retrieval (emergencies, personal advice) have no score and are left out of the choice.

**Rule** (`choose_threshold` in `src/evaluation.py`): the **lowest** threshold that keeps abstention recall at or above `min_abstention_recall = 0.90`. Balanced accuracy is reported for reference, but it is **not** the selection criterion.

In [ ]:
pipeline = load_pipeline()
print(f"{pipeline.retriever.index.ntotal:,} chunks")


def best_scores(df):
    """Best rerank score per question, without calling the LLM. NaN if the safety rules answer first."""
    scores = {}
    for row in df.itertuples():
        passages = [] if safety.check(row.question) else pipeline.retriever.retrieve(row.question)
        scores[row.id] = passages[0]["rerank_score"] if passages else np.nan
    return pd.Series(scores)


df_dev["best_score"] = df_dev["id"].map(best_scores(df_dev))
scored = df_dev.dropna(subset=["best_score"])
print(f"{len(df_dev) - len(scored)} of {len(df_dev)} dev questions never reach retrieval and are left out")

should_answer = scored[scored["expected_behavior"] == "answer"]
should_abstain = scored[scored["expected_behavior"] == "insufficient_evidence"]
calibration = choose_threshold(should_answer["best_score"], should_abstain["best_score"])
print(json.dumps(calibration, indent=2))

# Save the threshold so the API and the other notebooks use the same one
REPORTS.mkdir(exist_ok=True)
config.THRESHOLD_PATH.write_text(json.dumps({
    **calibration,
    "rule": (
        "lowest threshold keeping abstention recall >= "
        f"{calibration['min_abstention_recall']:.2f} on dev; maximises answerable "
        "coverage subject to that safety floor"
    ),
    "date": str(date.today()),
    "embedding_model": config.EMBEDDING_MODEL,
    "reranker_model": config.RERANKER_MODEL,
}, indent=2))
pipeline.threshold, pipeline.calibrated = calibration["threshold"], True

# Questions that fall on the wrong side of the chosen threshold
wrong = scored[
    ((scored["expected_behavior"] == "answer") & (scored["best_score"] < pipeline.threshold))
    | ((scored["expected_behavior"] == "insufficient_evidence") & (scored["best_score"] >= pipeline.threshold))
]
print(f"\nquestions on the wrong side of the threshold: {len(wrong)}")
wrong[["id", "type", "expected_behavior", "best_score", "question"]].round(2)

## 2. LLM-only baseline

The same generator model answers every question with no sources and no safety rules. It is judged exactly like the RAG answers.

This section makes a fresh set of LLM calls on every notebook run, so it is the first place
notebook 4 can hit a generator 429 (usually a tokens-per-day limit on the generator model).
Transient 429s are retried a bounded number of times by `llm_only_answers()` and `generate()`;
tokens-per-day exhaustion is raised immediately and clearly instead of being retried forever.
If that happens, rerun only this cell after the limit resets instead of running the whole
notebook again.

In [ ]:
def run_baseline(df):
    answers_df = llm_only_answers(client, df)
    answers = dict(zip(answers_df["id"], answers_df["answer"]))
    return {
        "answers": answers_df,
        "facts": score_key_facts(judge, df, answers),
        "behavior": score_behavior(judge, df, answers),
        "latency_ms": answers_df["latency_ms"].mean(),
    }


baseline_eval_dev = run_baseline(df_dev)
baseline_eval_dev["answers"].to_csv(REPORTS / "baseline_llm_only_dev.csv", index=False)
baseline_eval_dev["facts"].to_csv(REPORTS / "baseline_llm_only_dev_facts.csv", index=False)
baseline_eval_dev["behavior"].to_csv(REPORTS / "baseline_llm_only_dev_behavior.csv", index=False)

baseline_dev = summarize(baseline_eval_dev["facts"], baseline_eval_dev["behavior"], baseline_eval_dev["latency_ms"])
print("LLM-only baseline on dev, judged by", judge.model)
print(json.dumps(baseline_dev, indent=2))

## 3. Run the RAG pipeline on dev, then grade it

Every dev question goes through `pipeline.answer()`, one at a time so the latency is honest, with the threshold from section 1. The judge then grades key-fact coverage and behaviour. A question the pipeline declines counts as missing every key fact.

In [ ]:
def run_rag(df):
    results = {}
    for row in df.itertuples():
        try:
            results[row.id] = pipeline.answer(row.question)
        except Exception as error:
            print(f"{row.id}: {error}")
            results[row.id] = {"answer": "", "abstained": False, "emergency": False, "redirected": False,
                               "passages": [], "sources": [], "top_score": None, "latency_ms": None, "error": str(error)}
    return results


def is_declined(result):
    return result["abstained"] or result["redirected"] or result["emergency"]


def evaluate_rag(df, results):
    answers = {qid: r["answer"] for qid, r in results.items()}
    latencies = [r["latency_ms"] for r in results.values() if r["latency_ms"] is not None]
    return {
        "facts": score_key_facts(judge, df, answers),
        "behavior": score_behavior(judge, df, answers),
        "abstention": abstention_metrics(df["expected_behavior"].tolist(), [is_declined(results[q]) for q in df["id"]]),
        "latency_ms": float(np.mean(latencies)) if latencies else float("nan"),
    }


    rag_results_dev = run_rag(df_dev)
    rag_eval_dev = evaluate_rag(df_dev, rag_results_dev)

# Save per-question artifacts for auditability
pd.DataFrame([
    {"id": qid, "abstained": r["abstained"], "emergency": r["emergency"], "redirected": r["redirected"],
     "top_score": r["top_score"], "latency_ms": r["latency_ms"], "answer": r["answer"],
     "sources": json.dumps(r["sources"]), "error": r.get("error", "")}
    for qid, r in rag_results_dev.items()
]).to_csv(REPORTS / "rag_dev_answers.csv", index=False)
rag_eval_dev["facts"].to_csv(REPORTS / "rag_dev_facts.csv", index=False)
rag_eval_dev["behavior"].to_csv(REPORTS / "rag_dev_behavior.csv", index=False)

print("abstention:", rag_eval_dev["abstention"])

## 4. RAG vs LLM-only (dev)

The abstention threshold was tuned on this split, so abstention numbers here are optimistic.

In [ ]:
rag_dev = summarize(rag_eval_dev["facts"], rag_eval_dev["behavior"], rag_eval_dev["latency_ms"])
dev_table = comparison_table(baseline_dev, rag_dev, rag_eval_dev["abstention"])
print(f"Dev split ({len(df_dev)} questions; the threshold was tuned on it)")
dev_table

## 5. Coverage by question type (dev)

Where does the system decline, and where is it less complete than the LLM alone? This is the knowledge-base coverage check against the intended scope.
Consumer and research questions are what the sources are made for. Medication questions are out of scope, so declining them is the correct behaviour, not a gap.

In [ ]:
declined = df_dev["id"].map(lambda q: is_declined(rag_results_dev[q]))
by_type = pd.DataFrame({
    "questions": df_dev.groupby("type").size(),
    "RAG declined": declined.groupby(df_dev["type"]).mean(),
    "key-fact coverage, LLM-only": pd.Series(baseline_dev["coverage_by_type"]),
    "key-fact coverage, RAG": pd.Series(rag_dev["coverage_by_type"]),
}).round(2)
by_type

## 6. Read the failures

Numbers hide the interesting cases. Each one below is a retrieval problem, a threshold problem, a safety-rule problem, a prompt problem or a judge error.

In [ ]:
# Behaviour checks that failed (e.g. emergency not redirected, dose given)
failed = rag_eval_dev["behavior"][~rag_eval_dev["behavior"]["ok"]]
print(f"failed behaviour checks: {len(failed)}")
failed[["id", "type", "expected", "reason"]]

In [ ]:
# Answerable questions the system declined (over-refusal): weak evidence, or a safety rule that matched too much
refused = df_dev[(df_dev["expected_behavior"] == "answer") & declined]
print(f"answerable questions the system declined: {len(refused)}")
refused.assign(outcome=refused["id"].map(lambda q: "redirected" if rag_results_dev[q]["redirected"] else "emergency" if rag_results_dev[q]["emergency"] else "abstained"))[["id", "type", "outcome", "question"]]

## 7. Final test run (once)

Run this **once**, after you are happy with everything above: set `RUN_TEST = True` in the setup cell and run the notebook again from the top. It evaluates the untouched test split. The threshold is computed from the same dev scores again, so it stays frozen as long as nothing else changed.
If you change anything afterwards, the test split is no longer a fair check.

In [ ]:
if RUN_TEST:
    rag_results_test = run_rag(df_test)
    rag_eval_test = evaluate_rag(df_test, rag_results_test)
    rag_test = summarize(rag_eval_test["facts"], rag_eval_test["behavior"], rag_eval_test["latency_ms"])

    baseline_eval_test = run_baseline(df_test)
    baseline_test = summarize(baseline_eval_test["facts"], baseline_eval_test["behavior"], baseline_eval_test["latency_ms"])

    test_table = comparison_table(baseline_test, rag_test, rag_eval_test["abstention"])
    rag_eval_test["behavior"].to_csv(REPORTS / "rag_test_behavior.csv", index=False)
    print(f"Test split ({len(df_test)} questions, threshold frozen)")
    test_table
else:
    print("Test split not run (RUN_TEST = False).")

## 8. Write `reports/results.md`

Everything in the report is computed above. Nothing is typed in by hand.

In [ ]:
n_emergency = sum(r["emergency"] for r in rag_results_dev.values())
n_redirected = sum(r["redirected"] for r in rag_results_dev.values())
test_status = "evaluated once" if RUN_TEST else "not evaluated yet"

lines = [
    "# Evaluation results", "",
    f"Generated {date.today()} by `notebooks/04_evaluation.ipynb`. Every number below was computed by that notebook.", "",
    "## Setup", "",
    f"- Generator `{config.LLM_MODEL}`, judge `{judge.model}` (a different model)",
    f"- Embeddings `{config.EMBEDDING_MODEL}`, reranker `{config.RERANKER_MODEL}`",
    f"- Knowledge base: MedQuAD + MedlinePlus, {pipeline.retriever.index.ntotal:,} chunks",
    f"- Abstention threshold {pipeline.threshold:.2f}, calibrated on {len(scored)} dev questions that reach retrieval",
    f"- Dev questions: {len(df_dev)} ({n_emergency} answered by the emergency rule, {n_redirected} by the personal-advice redirect). "
    f"Test questions: {len(df_test)} ({test_status})", "",
    "## LLM-only vs RAG (dev)", "",
    "Both systems answered, and were judged on, the same questions with the same labels and key facts. "
    "The abstention threshold was tuned on this split, so abstention numbers are optimistic.", "",
    md_table(dev_table, "Metric"), "",
    "## Coverage by question type (dev)", "",
    "Medications are out of scope: the knowledge base has no drug source.", "",
    md_table(by_type.fillna("-"), "Type"), "",
]

if RUN_TEST:
    lines += ["## LLM-only vs RAG (test, threshold frozen)", "", md_table(test_table, "Metric"), ""]

lines += [
    "## Limitations", "",
    "- Key facts were written by the project owner and have not been reviewed by a clinician.",
    "- Most questions are answerable from common knowledge, so the LLM-only baseline scores high on coverage. The value of RAG here is grounding, citations and abstention.",
    f"- Only {len(should_abstain)} dev questions that reach retrieval should be abstained on, so the abstention threshold is a noisy estimate.",
    "- Answers are graded by an LLM. Every verdict is saved in `reports/*.csv`.",
    "- The safety rules were written while looking at some of these questions, so the emergency and redirect numbers are optimistic.",
    "- MedQuAD is 2019 data, so some content is dated.",
]
(REPORTS / "results.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("wrote", (REPORTS / "results.md").relative_to(ROOT))